# Multiple Regression with a Hidden Layer

## Multiple Inputs and a Hidden Layer

$$
\underbrace{\hat{y}}_{1 \times 1} = \text{ReLU}(\underbrace{x}_{1 \times p} \underbrace{W_h}_{p \times k} + \underbrace{b_h}_{1 \times k}) \underbrace{W_o}_{k \times 1} + \underbrace{b_o}_{1 \times 1}
$$

## Gradients for Multiple Inputs

$$
\frac{\partial L}{\partial w_{h,jm}} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) \, w_{o,m} \, \text{ReLU}'(h_{im}) \, x_{ij}
$$

$$
\underbrace{\frac{\partial L}{\partial W_o}}_{k \times 1} = -\frac{2}{n} \underbrace{Z^\top}_{k \times n} \underbrace{(y - \hat{y})}_{n \times 1}
\qquad
\underbrace{\frac{\partial L}{\partial b_o}}_{1 \times 1} = -\frac{2}{n} \underbrace{\mathbf{1}^\top}_{1 \times n} (y - \hat{y})
$$

$$
\underbrace{\frac{\partial L}{\partial W_h}}_{p \times k} = -\frac{2}{n} \underbrace{X^\top}_{p \times n} \Delta
\qquad
\underbrace{\frac{\partial L}{\partial b_h}}_{1 \times k} = -\frac{2}{n} \underbrace{\mathbf{1}^\top}_{1 \times n} \Delta
$$

## Implementation

### A hidden layer with two inputs

In [ ]:
import numpy as np
import torch
import torch.nn as nn

rng = np.random.default_rng(1)

n = 300
X_two = rng.uniform(-2, 2, size=(n, 2))
y_two = np.sin(2 * X_two[:, [0]]) + X_two[:, [1]] ** 2 + rng.normal(0, 0.1, size=(n, 1))

def relu(H):
    return np.maximum(0, H)

def relu_prime(H):
    return (H > 0).astype(float)

def train_relu_network(X, y, k=16, learning_rate=0.05, epochs=3000, seed=0):
    rng = np.random.default_rng(seed)
    n, p = X.shape

    W_h = rng.normal(0, np.sqrt(2 / p), size=(p, k))
    b_h = np.zeros((1, k))
    W_o = rng.normal(0, np.sqrt(2 / k), size=(k, 1))
    b_o = np.zeros((1, 1))

    for epoch in range(epochs):
        H = X @ W_h + b_h
        Z = relu(H)
        y_hat = Z @ W_o + b_o

        error = y - y_hat
        dW_o = -(2 / n) * Z.T @ error
        db_o = -(2 / n) * error.sum(keepdims=True)
        Delta = (error @ W_o.T) * relu_prime(H)
        dW_h = -(2 / n) * X.T @ Delta
        db_h = -(2 / n) * Delta.sum(axis=0, keepdims=True)

        W_o -= learning_rate * dW_o
        b_o -= learning_rate * db_o
        W_h -= learning_rate * dW_h
        b_h -= learning_rate * db_h

    return np.mean((y - (relu(X @ W_h + b_h) @ W_o + b_o)) ** 2)

# Compare with the best linear model
design_two = np.c_[np.ones(n), X_two]
linear_fit = design_two @ np.linalg.lstsq(design_two, y_two, rcond=None)[0]

print("MSE of multiple linear regression:", round(float(np.mean((y_two - linear_fit) ** 2)), 4))
print("MSE of the network with a hidden layer (NumPy):", round(float(train_relu_network(X_two, y_two)), 4))

In [ ]:
torch.manual_seed(0)

X_two_t = torch.from_numpy(X_two).float()
y_two_t = torch.from_numpy(y_two).float()

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(3000):
    loss = criterion(model(X_two_t), y_two_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("MSE of the network with a hidden layer (PyTorch):", round(loss.item(), 4))

## Multiple Linear Regression Without a Hidden Layer

$$
\hat{y} = X W + b
$$

### Gradients of the multiple regression model

$$
\frac{\partial L}{\partial w_j} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) x_{ij}
\qquad
\frac{\partial L}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)
$$

$$
\underbrace{\frac{\partial L}{\partial W}}_{p \times 1} = -\frac{2}{n} \underbrace{X^\top}_{p \times n} \underbrace{(y - \hat{y})}_{n \times 1}
\qquad
\underbrace{\frac{\partial L}{\partial b}}_{1 \times 1} = -\frac{2}{n} \underbrace{\mathbf{1}^\top}_{1 \times n} \underbrace{(y - \hat{y})}_{n \times 1}
$$

### Gradient descent versus OLS

In [ ]:
rng = np.random.default_rng(42)

n = 200
p = 2
X_lin = rng.normal(size=(n, p))                                                                   # n x p
y_lin = 1 + 2 * X_lin[:, [0]] - 3 * X_lin[:, [1]] + rng.normal(0, 0.5, size=(n, 1))               # n x 1

# Initialize and train with plain full-batch gradient descent
W = np.zeros((p, 1))
b = np.zeros((1, 1))
learning_rate = 0.1

for epoch in range(500):
    y_hat = X_lin @ W + b
    error = y_lin - y_hat

    dW = -(2 / n) * X_lin.T @ error                  # p x 1
    db = -(2 / n) * error.sum(keepdims=True)         # 1 x 1

    W -= learning_rate * dW
    b -= learning_rate * db

# Ordinary least squares (add a column of ones for the intercept)
design = np.c_[np.ones(n), X_lin]
ols_coefficients = np.linalg.lstsq(design, y_lin, rcond=None)[0].ravel()

print("Gradient descent: b =", b.ravel().round(4), " W =", W.ravel().round(4))
print("OLS:              b =", ols_coefficients[:1].round(4), " W =", ols_coefficients[1:].round(4))